In [1]:
%pip install pyswip

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyswip import Prolog

prolog = Prolog()
prolog.assertz("sviluppato_da(gioco1, minori)")
print(list(prolog.query("sviluppato_da(gioco1, X)")))

[{'X': 'minori'}]


In [3]:
import pandas as pd
from pathlib import Path

giochi = pd.read_pickle("../data/games.pkl")

data = pd.to_datetime(giochi["Release date"], format="mixed", errors="coerce")
giochi["data_num"] = data.dt.year * 10000 + data.dt.month * 100 + data.dt.day
print(len(giochi), "giochi,", giochi["data_num"].isna().sum(), "date mancanti")

125855 giochi, 0 date mancanti


In [4]:
def atomo(testo):
    testo = str(testo).strip().replace("\\", "\\\\").replace("'", "\\'")
    return f"'{testo}'"

def separa(testo):
    if pd.isna(testo):
        return []
    return [p.strip() for p in str(testo).split(",") if p.strip() != ""]

In [5]:
Path("../kb").mkdir(exist_ok=True)

with open("../kb/fatti.pl", "w", encoding="utf-8") as f:
    f.write(":- encoding(utf8).\n\n")

    for id_gioco, d in zip(giochi["AppID"], giochi["data_num"]):
        if pd.notna(d):
            f.write(f"uscita({id_gioco}, {int(d)}).\n")

    for id_gioco, devs in zip(giochi["AppID"], giochi["Developers"]):
        for dev in separa(devs):
            f.write(f"sviluppato_da({id_gioco}, {atomo(dev)}).\n")

    for id_gioco, pubs in zip(giochi["AppID"], giochi["Publishers"]):
        for pub in separa(pubs):
            f.write(f"pubblicato_da({id_gioco}, {atomo(pub)}).\n")

In [6]:
from pyswip import Prolog

prolog = Prolog()
prolog.consult("../kb/fatti.pl")

print(list(prolog.query("uscita(496350, Data)")))
print(list(prolog.query("sviluppato_da(496350, Dev)")))
print(list(prolog.query("pubblicato_da(496350, Pub)")))

[{'Data': 20160729}]
[{'Dev': 'minori'}]
[{'Pub': 'MangaGamer'}]


In [7]:
prolog.consult("../kb/regole.pl")

for q in ["n_giochi_precedenti_dev(496350, N)",
          "n_giochi_precedenti_pub(496350, N)",
          "autopubblicato(496350)"]:
    print(q, "->", list(prolog.query(q)))

print(list(prolog.query("autopubblicato(G)", maxresult=3)))

n_giochi_precedenti_dev(496350, N) -> [{'N': 1}]
n_giochi_precedenti_pub(496350, N) -> [{'N': 14}]
autopubblicato(496350) -> []
[{'G': 3292190}, {'G': 3631080}, {'G': 1654170}]


In [8]:
finale = pd.read_pickle("../data/dataset_finale.pkl")
id_giochi = giochi.loc[finale.index, "AppID"]

def chiedi_numero(query):
    risposta = list(prolog.query(query))
    return risposta[0]["N"]

esp_dev, esp_pub, auto = [], [], []

for i, g in enumerate(id_giochi):
    esp_dev.append(chiedi_numero(f"n_giochi_precedenti_dev({g}, N)"))
    esp_pub.append(chiedi_numero(f"n_giochi_precedenti_pub({g}, N)"))
    auto.append(1 if list(prolog.query(f"autopubblicato({g})", maxresult=1)) else 0)
    if i % 10000 == 0:
        print(i, "giochi elaborati")

finale["esperienza_dev"] = esp_dev
finale["esperienza_pub"] = esp_pub
finale["autopubblicato"] = auto
print(finale.shape)

0 giochi elaborati
10000 giochi elaborati
20000 giochi elaborati
30000 giochi elaborati
40000 giochi elaborati
50000 giochi elaborati
(56662, 26)


In [9]:
print(finale[["esperienza_dev", "esperienza_pub", "autopubblicato"]].describe().T)
print()
print(finale.groupby("autopubblicato")["apprezzato"].mean())
print()
fasce = pd.qcut(finale["esperienza_dev"], 4, duplicates="drop")
print(finale.groupby(fasce, observed=True)["apprezzato"].mean())

finale.to_pickle("../data/dataset_kb.pkl")

                  count       mean        std  min  25%  50%  75%    max
esperienza_dev  56662.0   6.971480  33.446662  0.0  0.0  0.0  2.0  752.0
esperienza_pub  56662.0  15.227242  48.733318  0.0  0.0  1.0  7.0  977.0
autopubblicato  56662.0   0.702605   0.457116  0.0  0.0  1.0  1.0    1.0

autopubblicato
0    0.500920
1    0.549396
Name: apprezzato, dtype: float64

esperienza_dev
(-0.001, 2.0]    0.543182
(2.0, 752.0]     0.507927
Name: apprezzato, dtype: float64
